# 08 — Final verification

Run this notebook from a fresh kernel. It reads only what is on disk.

Every check appends one row to a results table with a `PASS` / `FAIL` / `WARN` status. A check that
raises is recorded as `FAIL` with the exception text, and the final cell asserts that there are no
`FAIL` rows.

Checks performed:

1. all upstream success markers and hashes present and matching
2. all 24 patient runs complete; 35 CellPhoneDB runs in total (27 core + 8 pooled seed)
3. patient labels derive only from `clinical_outcome`
4. score extraction preserved structural missingness
5. every tested endpoint meets the 6/6 evaluability and prevalence rules
6. Benjamini–Hochberg applied to the complete estimable family, once per family
7. pooled significance was not used to define the primary family
8. effect-size signs agree with medians, or the tie case is documented
9. confidence intervals and permutation results within expected bounds
10. all output tables have unique keys
11. figures regenerate from saved tables
12. config, database, inputs, code and output checksums recorded
13. CellPhoneDB interaction coverage recorded

Outputs: `FINAL_VERIFICATION.md`, `final_verification_checks.csv`, `MANIFEST.sha256`.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
# =============================================================================
# CONFIGURATION — every analytical value is read from config.yaml, the single
# source of truth whose SHA-256 is stamped into every success record.
# =============================================================================
from pathlib import Path
import sys

NOTEBOOK = "08_final_verification"

CODE_DIR    = Path(os.getcwd())
CONFIG_PATH = CODE_DIR / "config.yaml"

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from utils import load_config                                        # noqa: E402

CFG = load_config(CONFIG_PATH)

PROJECT_ROOT   = Path(CFG["project_root"])
OUTPUT_ROOT    = Path(CFG["output_root"])
assert Path(CFG["code_root"]) == CODE_DIR, \
    f"config code_root {CFG['code_root']} != {CODE_DIR}"

CPDB_RUNS       = OUTPUT_ROOT / "cpdb_runs"
ALL_SAMPLES_DIR = CPDB_RUNS / "all"
POOLED_DIR      = CPDB_RUNS / "pooled"
PER_PATIENT_DIR = CPDB_RUNS / "per_patient"
ALL_POOLED_SEEDS = [int(s) for s in CFG["pooled_downsample_seeds"]]
POOLED_SEEDS     = [s for s in ALL_POOLED_SEEDS if s != 0]   # seed 0 is the main run

PRIMARY_DIR         = OUTPUT_ROOT / "primary"
PRIMARY_RESULTS_CSV = PRIMARY_DIR / "patient_level_primary_results.csv"
EXPLORATORY_DIR     = OUTPUT_ROOT / "exploratory_pooled"
DE_RESULTS_DIR      = EXPLORATORY_DIR / "differential_enrichment_results"
CONSISTENCY_DIR     = EXPLORATORY_DIR / "consistency_results"
SENSITIVITY_DIR     = OUTPUT_ROOT / "sensitivity"
FIG_ROOT            = OUTPUT_ROOT / "figures"
TABLES_DIR          = OUTPUT_ROOT / "tables"
LOG_DIR             = OUTPUT_ROOT / "logs"
AUDIT_DIR           = OUTPUT_ROOT / "audit"

# Interaction-coverage artifacts produced by cpdb_interaction_coverage.py.
# The numbers are READ from these files, never retyped into this notebook.
COVERAGE_JSON      = AUDIT_DIR / "cpdb_interaction_coverage.json"
DETECTABILITY_CSV  = AUDIT_DIR / "cpdb_interaction_detectability.csv"

# The ACC tier-1 communication axes, as EXACT CellPhoneDB classification names.
# Exact names are used here so the coverage table is independent of any
# substring matching.
ACC_COVERAGE_CLASSIFICATIONS = [
    "Signaling by Notch",
    "Signaling by Fibroblast growth factor",
    "Signaling by Transforming growth factor",
    "Signaling by Vascular endothelial growth factor",
    "Signaling by Epidermal growth factor",
    "Signaling by Ephrin",
    "Signaling by Tumor necrosis factor",
    "Signaling by Chemokines",
    "Signaling by Interleukin",
    "Signaling by Integrin",
    "Adhesion by Collagen/Integrin",
    "Signaling by WNT",
    "Signaling by WNT inhibition",
]
# The pair that can manufacture a confident but false biological narrative.
WNT_ARTIFACT_PAIR = ("Signaling by WNT", "Signaling by WNT inhibition")

RUN_LOG          = LOG_DIR / f"{NOTEBOOK}_run_log.jsonl"
VERIFICATION_MD  = OUTPUT_ROOT / "FINAL_VERIFICATION.md"
VERIFICATION_CSV = OUTPUT_ROOT / "final_verification_checks.csv"
MANIFEST_PATH    = OUTPUT_ROOT / "MANIFEST.sha256"
SUCCESS_JSON     = OUTPUT_ROOT / "08_SUCCESS.json"

# --- Expected structure -----------------------------------------------------
# Cohort counts are dataset facts asserted in 00_audit_inputs_and_sources, not
# configuration keys: config.yaml holds no cohort counts. The run
# counts below are derived from the configured seed list.
EXPECTED_N_PATIENTS  = 24
EXPECTED_N_GOOD      = 13
EXPECTED_N_POOR      = 11
EXPECTED_N_CELLTYPES = 19
EXPECTED_N_CELLS     = 95441
assert EXPECTED_N_GOOD + EXPECTED_N_POOR == EXPECTED_N_PATIENTS

EXPECTED_CORE_RUNS   = EXPECTED_N_PATIENTS + 3       # per-patient + pooled good/poor + all
EXPECTED_SEED_RUNS   = len(POOLED_SEEDS) * 2         # stability seeds x 2 outcome groups
EXPECTED_TOTAL_RUNS  = EXPECTED_CORE_RUNS + EXPECTED_SEED_RUNS   # 27 + 8 = 35
assert (EXPECTED_CORE_RUNS, EXPECTED_SEED_RUNS, EXPECTED_TOTAL_RUNS) == (27, 8, 35)

REQUIRED_CPDB_OUTPUTS = ["interaction_scores.csv", "means.csv",
                         "pvalues.csv", "significant_means.csv"]

# --- Statistical rules — read from config.yaml, never re-declared -----------
MIN_EVALUABLE_PER_GROUP   = int(CFG["min_evaluable_per_group"])
MIN_NONZERO_TOTAL         = int(CFG["min_nonzero_total"])
FDR_ALPHA                 = float(CFG["fdr_alpha"])
FDR_METHOD                = CFG["fdr_method"]
PATIENT_CELLTYPE_MIN_CELLS = int(CFG["patient_celltype_min_cells"])
MANNWHITNEY_METHOD        = CFG["mannwhitney_method"]
OUTCOME_COL               = CFG["outcome_column"]
OUTCOME_LABELS            = list(CFG["outcomes"])
BOOTSTRAP_RESAMPLES       = int(CFG["bootstrap_resamples"])
PERMUTATION_RESAMPLES     = int(CFG["permutation_resamples"])
ANALYSIS_SEED             = int(CFG["analysis_seed"])

# The design values locked at prespecification. C14 asserts config.yaml still
# carries these, so that editing the config after the fact is caught rather
# than silently honoured.
PRESPECIFIED_DESIGN = {
    "min_evaluable_per_group": 6,
    "min_nonzero_total": 6,
    "fdr_alpha": 0.05,
    "fdr_method": "fdr_bh",
    "outcome_column": "clinical_outcome",
    "patient_celltype_min_cells": 10,
    "mannwhitney_method": "hybrid",
    "bootstrap_resamples": 10000,
    "permutation_resamples": 99999,
    "analysis_seed": 20260731,
    "pooled_downsample_seeds": [0, 1, 2, 3, 4],
    "outcomes": ["Good", "Poor"],
}

# --- Notebooks that must exist and must not read batch_1 --------------------
NOTEBOOK_NAMES = [
    "00_audit_inputs_and_sources.ipynb",
    "01_build_cpdb_inputs.ipynb",
    "02_run_cellphonedb.ipynb",
    "03_extract_patient_endpoints.ipynb",
    "04_patient_level_inference.ipynb",
    "05_pooled_exploratory_analysis.ipynb",
    "06_sensitivity_analysis.ipynb",
    "07_figures_and_tables.ipynb",
    "08_final_verification.ipynb",
]
FORBIDDEN_OBS_COLUMN = "batch_1"
# config.yaml carries forbidden_obs_columns; this literal is a genuine fallback
# used only if the key is ever removed from the config.
FORBIDDEN_OBS_COLUMNS_FALLBACK = ["batch_1", "site", "model_outcome", "outcome"]
FORBIDDEN_OBS_COLUMNS = list(CFG.get("forbidden_obs_columns")
                             or FORBIDDEN_OBS_COLUMNS_FALLBACK)
assert FORBIDDEN_OBS_COLUMN in FORBIDDEN_OBS_COLUMNS


# --- Hybrid Mann-Whitney justification constants ----------------------------
# Perfect-separation p-value floors (scipy.stats.mannwhitneyu, two-sided). The
# two endpoint families have DIFFERENT group sizes and therefore different
# floors -- using the primary floor for the secondary family understates the
# requirement by a factor of five:
#   primary   directional endpoints, all 24 patients ......... 13 Good / 11 Poor
#   secondary interaction-level over the fixed 36-pair set,
#             which only 20 of the 24 patients supply in full .. 11 Good /  9 Poor
#
# Benjamini-Hochberg rejects at rank k when p_(k) <= k*alpha/m, so the operative
# quantity is not "can one endpoint at the floor be rejected" but "how large a
# cluster of endpoints tied at the floor is rejectable":
#     k_min = ceil(m * floor / alpha)
# Verified empirically with statsmodels.stats.multitest.multipletests, alpha=0.05.
import math

MW_FLOORS = {
    "primary": {
        "label": "primary directional endpoints",
        "n_good": 13, "n_poor": 11,
        "asymptotic": 3.8971e-05, "exact": 8.0124e-07,
        "family_size": 20000,
        "min_cluster_asymptotic": 16,   # 15 at the floor reject 0; 16 reject all 16
        "min_cluster_exact": 1,
    },
    "secondary": {
        "label": "secondary interaction-level endpoints",
        "n_good": 11, "n_poor": 9,
        "asymptotic": 1.9709e-04, "exact": 1.1908e-05,
        "family_size": 1000,
        "min_cluster_asymptotic": 4,    # 3 at the floor reject 0; 4 reject all 4
        "min_cluster_exact": 1,
    },
}
for _fam in MW_FLOORS.values():
    assert math.ceil(_fam["family_size"] * _fam["asymptotic"] / 0.05) \
        == _fam["min_cluster_asymptotic"], f"cluster arithmetic disagrees for {_fam['label']}"
    assert _fam["asymptotic"] > _fam["exact"]

MW_ASYMPTOTIC_PERFECT_SEPARATION = MW_FLOORS["primary"]["asymptotic"]
MW_EXACT_PERFECT_SEPARATION      = MW_FLOORS["primary"]["exact"]

# --- Pooled balanced design: retention boundary ------------------------------
# Rule 2 is min(n_good, n_poor) > 10 -- STRICTLY greater than
# config['pooled_celltype_min_per_group'], not >=. The boundary case is
# Plasma cells at exactly 10 Good / 157 Poor: retaining it would balance the
# comparison down to 10 cells per group. Both plan documents were internally
# inconsistent here (the >= rule keeps Plasma cells, giving 16 types and 42,224
# per group, yet both quoted 15 types / 42,214 and listed Plasma cells as
# dropped). The resolution is > 10; Plasma cells is dropped.
POOLED_MIN_PER_GROUP_STRICT     = True
POOLED_EXPECTED_RETAINED_TYPES  = 15
POOLED_EXPECTED_CELLS_PER_GROUP = 42214
POOLED_EXPECTED_TOTAL_CELLS     = 84428
POOLED_DROPPED_TYPES = {                  # cell type -> (n_good, n_poor)
    "Basal cells - Tumor":             (276, 0),
    "Fibroblast cells - CAF":          (790, 6),
    "Fibroblast cells - Inflammatory": (145, 7),
    "Plasma cells":                    (10, 157),
}
assert POOLED_EXPECTED_CELLS_PER_GROUP * 2 == POOLED_EXPECTED_TOTAL_CELLS
assert EXPECTED_N_CELLTYPES - len(POOLED_DROPPED_TYPES) == POOLED_EXPECTED_RETAINED_TYPES
assert all(min(g, p) <= int(CFG["pooled_celltype_min_per_group"])
           for g, p in POOLED_DROPPED_TYPES.values()), \
    "a cell type listed as dropped satisfies the > 10 retention rule"


# CellPhoneDB rounds interaction scores to 3 decimals inside
# cellphonedb/utils/scoring_utils.py (np.around(lr_outer, 3)). result_precision
# does NOT reach interaction_scores; it affects means, significant_means and
# deconvoluted only.
INTERACTION_SCORE_DECIMALS_HARDCODED = 3

LOG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# =============================================================================
# IMPORTS + check harness
# =============================================================================
import hashlib
import json
import os
import re
import subprocess
import sys
import time
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# CFG was loaded in the configuration cell; only the remaining helpers are
# imported here. Nothing in this cell re-reads or re-declares configuration.
from utils import (sha256_file, write_manifest, log_jsonl,           # noqa: E402
                   derive_patient_outcomes, safe_unlink)
try:                                    # the per-run completion marker filename
    from utils import SUCCESS_FILENAME  # noqa: E402
except ImportError:
    SUCCESS_FILENAME = "_SUCCESS.json"


def _call_flex(fn, *variants):
    last = None
    for args, kwargs in variants:
        try:
            return fn(*args, **kwargs)
        except TypeError as exc:
            last = exc
    raise last


CFG_SHA = sha256_file(CONFIG_PATH)

RUN_START = time.time()
RUN_START_ISO = datetime.now(timezone.utc).isoformat()

CHECKS = []
MANIFEST_ENTRIES = {}      # populated by check C13


def record(check_id, name, status, detail, evidence=None):
    row = {"check_id": check_id, "name": name, "status": status,
           "detail": str(detail), "evidence": json.dumps(evidence, default=str)
           if evidence is not None else ""}
    CHECKS.append(row)
    _call_flex(log_jsonl,
               ((RUN_LOG, {"notebook": NOTEBOOK, "event": "check", **row}), {}),
               (({"notebook": NOTEBOOK, "event": "check", **row}, RUN_LOG), {}))
    print(f"[{status:4}] {check_id}  {name} - {detail}")
    return row


def run_check(check_id, name):
    '''Decorator: a check that raises is recorded FAIL, never aborts the notebook.'''
    def deco(fn):
        try:
            status, detail, evidence = fn()
        except Exception as exc:                                   # noqa: BLE001
            status, detail, evidence = "FAIL", f"{type(exc).__name__}: {exc}", None
        record(check_id, name, status, detail, evidence)
        return fn
    return deco


def read_any(path):
    path = Path(path)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


def find_first(patterns, roots=None):
    roots = roots or [OUTPUT_ROOT]
    for pat in np.atleast_1d(patterns):
        for root in roots:
            hits = sorted(p for p in Path(root).glob(str(pat)) if p.is_file())
            if hits:
                return hits[0]
    return None


def coverage_by_classification():
    '''Per-classification detectability, grouped from the audit artifact.

    Returns classification / detectable / total / pct. Interactions with no
    classification are bucketed as "(unclassified)" so the totals still
    reconcile to the database size.
    '''
    d = pd.read_csv(DETECTABILITY_CSV)
    d["detectable"] = d["detectable"].astype(bool)
    d["classification"] = (d["classification"].fillna("(unclassified)")
                           .astype(str).str.strip())
    g = (d.groupby("classification")["detectable"]
           .agg(detectable="sum", total="size").reset_index())
    g["detectable"] = g["detectable"].astype(int)
    g["pct"] = 100.0 * g["detectable"] / g["total"]
    return g.sort_values(["detectable", "total"], ascending=[True, False])


def col(df, candidates, required=True):
    lower = {c.lower(): c for c in df.columns}
    for cand in np.atleast_1d(candidates):
        if cand in df.columns:
            return cand
        if cand.lower() in lower:
            return lower[cand.lower()]
    assert not required, f"none of {list(np.atleast_1d(candidates))} in {list(df.columns)}"
    return None


# --- Workflow start timestamp, taken from the upstream success markers ------
# Notebook-level markers use several naming conventions upstream
# (gate1_success.json, 02_success_record.json, 05_SUCCESS.json, ...), and the
# per-run markers are utils.SUCCESS_FILENAME. Collect them all, case-insensitively.
SUCCESS_FILES = sorted({p for p in OUTPUT_ROOT.rglob("*.json")
                        if "success" in p.name.lower()
                        and p.name != SUCCESS_FILENAME
                        # This notebook's own marker is not an upstream artifact.
                        and p != SUCCESS_JSON})
_starts = []
for p in SUCCESS_FILES:
    try:
        d = json.loads(p.read_text())
        for k in ("run_start_utc", "start_time", "started_utc", "start_utc"):
            if isinstance(d, dict) and d.get(k):
                _starts.append(str(d[k]))
                break
    except Exception:
        continue

if _starts:
    WORKFLOW_START_ISO = min(_starts)
elif hasattr(CFG, "get") and CFG.get("workflow_start_utc"):
    WORKFLOW_START_ISO = str(CFG["workflow_start_utc"])
else:
    _mt = min((p.stat().st_mtime for p in OUTPUT_ROOT.rglob("*") if p.is_file()),
              default=RUN_START)
    WORKFLOW_START_ISO = datetime.fromtimestamp(_mt, timezone.utc).isoformat()

WORKFLOW_START_TS = pd.Timestamp(WORKFLOW_START_ISO).tz_convert("UTC") \
    if pd.Timestamp(WORKFLOW_START_ISO).tzinfo else \
    pd.Timestamp(WORKFLOW_START_ISO).tz_localize("UTC")
WORKFLOW_START_EPOCH = WORKFLOW_START_TS.timestamp()
WORKFLOW_START_FIND = WORKFLOW_START_TS.strftime("%Y-%m-%d %H:%M:%S UTC")

print(f"workflow start (for modification checks): {WORKFLOW_START_FIND}")
print(f"success markers found: {len(SUCCESS_FILES)}")

In [ ]:
# =============================================================================
# CHECK 1 - upstream success markers and hashes present and matching
# =============================================================================
@run_check("C01", "upstream success markers and hashes present and matching")
def _c01():
    assert SUCCESS_FILES, "no *SUCCESS*.json markers found under the output root"
    markers, mismatches, missing_files, no_hashes = {}, [], [], []

    for p in SUCCESS_FILES:
        d = json.loads(p.read_text())
        markers[str(p.relative_to(OUTPUT_ROOT))] = d.get("status", "unknown")
        if d.get("status") not in (None, "success"):
            mismatches.append(f"{p.name}: status={d.get('status')}")

        hashes = d.get("output_sha256") or d.get("output_hashes") or {}
        # 03_ keys its hashes by logical name and carries a sibling name->path map
        path_map = d.get("outputs") or {}
        if not isinstance(path_map, dict):
            path_map = {}
        if not hashes:
            # 04_/05_/06_/07_ use written_files: {path: {"sha256": ..., ...}}
            wf = d.get("written_files") or {}
            if isinstance(wf, dict):
                hashes = {k: v.get("sha256") for k, v in wf.items()
                          if isinstance(v, dict) and v.get("sha256")}
        if not hashes:
            no_hashes.append(p.name)
        for fp, expected in hashes.items():
            f = Path(path_map.get(fp, fp))
            if not f.is_file():
                missing_files.append(fp)
                continue
            actual = sha256_file(f)
            if actual != expected:
                mismatches.append(f"{fp}: recorded {expected[:12]}... actual {actual[:12]}...")

    ev = {"markers": markers, "n_marker_files": len(SUCCESS_FILES),
          "markers_without_hashes": no_hashes,
          "missing_hashed_files": missing_files[:20],
          "hash_mismatches": mismatches[:20]}
    if mismatches or missing_files:
        return "FAIL", (f"{len(mismatches)} hash/status mismatches, "
                        f"{len(missing_files)} hashed files missing"), ev
    if no_hashes:
        return "WARN", f"{len(no_hashes)} success markers record no output hashes", ev
    return "PASS", f"{len(SUCCESS_FILES)} success markers, all recorded hashes match", ev

In [ ]:
# =============================================================================
# CHECK 3 - 24 patient runs complete; 35 CellPhoneDB runs in total
# =============================================================================
@run_check("C03", "24 patient runs complete; 35 runs total (27 core + 8 pooled seed)")
def _c03():
    def run_ok(d):
        d = Path(d)
        if not d.is_dir():
            return False, "directory missing"
        for f in REQUIRED_CPDB_OUTPUTS:
            p = d / f
            if not p.is_file() or p.stat().st_size == 0:
                return False, f"missing/empty {f}"
        marker = d / SUCCESS_FILENAME
        if not marker.is_file():
            return False, f"no {SUCCESS_FILENAME} completion marker"
        try:
            if json.loads(marker.read_text()).get("status") != "success":
                return False, "completion marker status is not 'success'"
        except Exception as exc:                                     # noqa: BLE001
            return False, f"unreadable completion marker: {exc}"
        return True, "ok"

    patient_dirs = sorted(p for p in PER_PATIENT_DIR.glob("*") if p.is_dir()) \
        if PER_PATIENT_DIR.is_dir() else []
    patient_status = {p.name: run_ok(p) for p in patient_dirs}
    bad_patients = {k: v[1] for k, v in patient_status.items() if not v[0]}

    core = [ALL_SAMPLES_DIR, POOLED_DIR / "good", POOLED_DIR / "poor"] + patient_dirs
    seeds = [CPDB_RUNS / f"pooled_seed{s}" / g for s in POOLED_SEEDS for g in ("good", "poor")]
    core_status = {str(d.relative_to(OUTPUT_ROOT)): run_ok(d) for d in core}
    seed_status = {str(d.relative_to(OUTPUT_ROOT)): run_ok(d) for d in seeds}

    n_core_ok = sum(1 for v in core_status.values() if v[0])
    n_seed_ok = sum(1 for v in seed_status.values() if v[0])
    ev = {"n_patient_dirs": len(patient_dirs),
          "bad_patient_runs": bad_patients,
          "n_core_complete": n_core_ok, "expected_core": EXPECTED_CORE_RUNS,
          "n_seed_complete": n_seed_ok, "expected_seed": EXPECTED_SEED_RUNS,
          "n_total_complete": n_core_ok + n_seed_ok,
          "expected_total": EXPECTED_TOTAL_RUNS,
          "incomplete_core": {k: v[1] for k, v in core_status.items() if not v[0]},
          "incomplete_seed": {k: v[1] for k, v in seed_status.items() if not v[0]}}

    problems = []
    if len(patient_dirs) != EXPECTED_N_PATIENTS:
        problems.append(f"{len(patient_dirs)} patient run directories, expected {EXPECTED_N_PATIENTS}")
    if bad_patients:
        problems.append(f"{len(bad_patients)} incomplete patient runs")
    if n_core_ok != EXPECTED_CORE_RUNS:
        problems.append(f"{n_core_ok}/{EXPECTED_CORE_RUNS} core runs complete")
    if n_seed_ok != EXPECTED_SEED_RUNS:
        problems.append(f"{n_seed_ok}/{EXPECTED_SEED_RUNS} pooled seed runs complete")
    if problems:
        return "FAIL", "; ".join(problems), ev
    return "PASS", (f"{EXPECTED_N_PATIENTS}/{EXPECTED_N_PATIENTS} patient runs; "
                    f"{n_core_ok + n_seed_ok}/{EXPECTED_TOTAL_RUNS} runs total"), ev

In [ ]:
# =============================================================================
# CHECK 4 - patient labels derive ONLY from clinical_outcome
# =============================================================================
# Detection is AST-based, not textual. A "read" is a genuine subscript or
# attribute access whose base expression is an obs table -- obs["site"],
# adata.obs["batch_1"], obs.model_outcome. Prose in comments, markdown,
# docstrings and log messages is invisible to the AST and is never a violation.
#
# 00_audit_inputs_and_sources.ipynb is exempt by design: its stated purpose is
# to read these columns once, document the discrepancy (batch_1 mislabels
# P07 and implies an incorrect 12/12 split; obs['site'] splits Base of
# Tongue into two outcome-confounded labels) and then reject them. Its reads
# are reported as audit reads. No other notebook may read them at all.
import ast

AUDIT_EXEMPT = {"00_audit_inputs_and_sources.ipynb"}
OBS_BASE = re.compile(r"(^|\.)obs(_\w+)?$")


def _is_obs_base(expr):
    '''True if an unparsed base expression refers to an obs table.'''
    expr = expr.strip()
    return bool(OBS_BASE.search(expr))


def ast_column_reads(src, names):
    '''Genuine attribute/subscript reads of `names`. None if src will not parse.'''
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return None
    hits = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Subscript):
            sl, targets = node.slice, []
            if isinstance(sl, ast.Constant) and isinstance(sl.value, str):
                targets = [sl.value]
            elif isinstance(sl, (ast.List, ast.Tuple)):
                targets = [e.value for e in sl.elts
                           if isinstance(e, ast.Constant) and isinstance(e.value, str)]
            for t in targets:
                if t in names:
                    hits.append({"name": t, "base": ast.unparse(node.value),
                                 "lineno": getattr(node, "lineno", 0)})
        elif isinstance(node, ast.Attribute) and node.attr in names:
            hits.append({"name": node.attr, "base": ast.unparse(node.value),
                         "lineno": getattr(node, "lineno", 0)})
    return hits


def iter_code_units(path):
    '''Yield (label, source) for each code unit in a notebook or .py file.'''
    if path.suffix == ".ipynb":
        nb = json.loads(path.read_text())
        for i, c in enumerate(nb.get("cells", [])):
            if c.get("cell_type") == "code":
                s = c["source"]
                yield f"cell{i}", (s if isinstance(s, str) else "".join(s))
    else:
        yield "module", path.read_text()


@run_check("C04", "patient labels come only from clinical_outcome; forbidden obs columns unread")
def _c04():
    files = [CODE_DIR / n for n in NOTEBOOK_NAMES]
    missing_nb = [f.name for f in files if not f.is_file()]
    scanned = [f for f in files if f.is_file()]
    utils_py = CODE_DIR / "utils.py"
    if utils_py.is_file():
        scanned.append(utils_py)

    # FORBIDDEN_OBS_COLUMNS comes from config.yaml (configuration cell).
    forbidden = set(FORBIDDEN_OBS_COLUMNS)
    from_config = bool(CFG.get("forbidden_obs_columns"))

    violations, audit_reads, non_obs, unparsed = [], [], [], []
    for f in scanned:
        for label, src in iter_code_units(f):
            hits = ast_column_reads(src, forbidden)
            if hits is None:
                unparsed.append(f"{f.name}:{label}")
                continue
            for h in hits:
                where = f"{f.name}:{label}:L{h['lineno']}  {h['base']}[{h['name']!r}]"
                if not _is_obs_base(h["base"]):
                    non_obs.append(where)          # e.g. a report dict keyed by the name
                elif f.name in AUDIT_EXEMPT:
                    audit_reads.append(where)
                else:
                    violations.append(where)

    outcome_col = OUTCOME_COL
    ev = {"n_files_scanned": len(scanned), "missing_notebooks": missing_nb,
          "forbidden_obs_columns": sorted(forbidden),
          "forbidden_list_source": "config.yaml" if from_config else "notebook fallback",
          "violations": violations,
          "audit_notebook_reads_exempt": audit_reads,
          "non_obs_uses_of_same_names": non_obs[:20],
          "unparseable_code_units": unparsed,
          "configured_outcome_column": outcome_col,
          "detection": "ast.Subscript / ast.Attribute with an obs-table base expression",
          "exempt_files": sorted(AUDIT_EXEMPT)}

    problems = []
    if missing_nb:
        problems.append(f"missing notebooks: {missing_nb}")
    if violations:
        problems.append(f"{len(violations)} obs reads of a forbidden column outside "
                        "the audit notebook")
    if outcome_col not in (None, "clinical_outcome"):
        problems.append(f"config outcome_column is {outcome_col!r}, not 'clinical_outcome'")
    if unparsed:
        problems.append(f"{len(unparsed)} code units could not be parsed and were not checked")
    if problems:
        return "FAIL", "; ".join(problems), ev
    return "PASS", (f"{len(scanned)} files scanned; 0 obs reads of {sorted(forbidden)} "
                    f"outside the audit notebook ({len(audit_reads)} documented audit reads)"), ev


@run_check("C04b", "derived patient split is 13 Good / 11 Poor from clinical_outcome")
def _c04b():
    gp = find_first(["**/patient_group_assignment.csv", "**/*patient_outcome*.csv",
                     "**/*derived_patient*.csv"])
    assert gp is not None, "no derived patient-outcome mapping found under the output root"
    df = read_any(gp)
    c_id = col(df, ["sample_id", "batch", "sample", "patient", "patient_id"])
    c_oc = col(df, ["clinical_outcome", "outcome", "group"])
    vc = df.drop_duplicates(subset=[c_id])[c_oc].value_counts().to_dict()
    ev = {"source": str(gp), "counts": vc}
    if vc.get("Good") != EXPECTED_N_GOOD or vc.get("Poor") != EXPECTED_N_POOR:
        return "FAIL", f"split is {vc}, expected 13 Good / 11 Poor", ev
    if set(vc) != {"Good", "Poor"}:
        return "FAIL", f"unexpected outcome labels: {sorted(vc)}", ev
    return "PASS", "13 Good / 11 Poor", ev

In [ ]:
# =============================================================================
# CHECK 5 - score extraction preserved structural missingness
# =============================================================================
@run_check("C05", "score extraction preserved structural missingness (never zeroed)")
def _c05():
    # The long score table holds only the evaluated rows by construction; the
    # availability table carries the full status classification, so prefer it.
    lp = find_first(["**/patient_endpoint_availability.parquet",
                     "**/patient_endpoint_availability.csv",
                     "**/patient_endpoint_scores_long.parquet",
                     "**/patient_endpoint_scores_long.csv",
                     "**/patient_endpoint_scores*.parquet",
                     "**/patient_endpoint_scores*.csv"])
    assert lp is not None, "patient endpoint availability/long table not found"
    df = read_any(lp)
    c_score = col(df, ["score_evaluated", "interaction_score", "score", "value"])
    c_status = col(df, ["availability", "status", "endpoint_status"], required=False)

    n_nan = int(df[c_score].isna().sum())
    ev = {"source": str(lp), "n_rows": int(len(df)), "n_nan_scores": n_nan}

    if c_status is None:
        if n_nan == 0:
            return "FAIL", ("no availability/status column and no NaN scores: "
                            "structural missingness cannot be shown to be preserved"), ev
        return "WARN", (f"no availability/status column; {n_nan} NaN scores present, "
                        "consistent with preserved missingness"), ev

    counts = df[c_status].value_counts().to_dict()
    ev["status_counts"] = counts
    sm = df[df[c_status].astype(str).str.lower().str.contains("structural")]
    ev["n_structural_missing"] = int(len(sm))
    if len(sm) == 0:
        return "WARN", "no rows classified structural_missing", ev
    zeroed = int((sm[c_score].fillna(-1) == 0).sum())
    nonnull = int(sm[c_score].notna().sum())
    ev["structural_rows_with_zero_score"] = zeroed
    ev["structural_rows_with_a_value"] = nonnull
    if zeroed > 0 or nonnull > 0:
        return "FAIL", (f"{zeroed} structural_missing rows hold 0.0 and {nonnull} hold a "
                        "numeric score: missingness was converted to a value"), ev
    return "PASS", (f"{len(sm)} structural_missing rows all NaN; "
                    f"status classes: {sorted(counts)}"), ev

In [ ]:
# =============================================================================
# Load the primary results once, for checks 6-11
# =============================================================================
PRIMARY = None
P = {}
try:
    PRIMARY = read_any(PRIMARY_RESULTS_CSV)
    P = {k: col(PRIMARY, v, required=req) for k, v, req in [
        ("id",      ["id_cp_interaction"], True),
        ("sender",  ["sender"], True),
        ("recv",    ["receiver"], True),
        ("ng",      ["n_good"], True),
        ("np_",     ["n_poor"], True),
        ("nzg",     ["nonzero_good"], False),
        ("nzp",     ["nonzero_poor"], False),
        ("medg",    ["median_good"], True),
        ("medp",    ["median_poor"], True),
        ("mdiff",   ["median_difference"], False),
        ("rb",      ["rank_biserial"], True),
        ("rblo",    ["rank_biserial_ci_low"], True),
        ("rbhi",    ["rank_biserial_ci_high"], True),
        ("mdlo",    ["median_difference_ci_low"], False),
        ("mdhi",    ["median_difference_ci_high"], False),
        ("p",       ["p_value"], True),
        ("q",       ["q_value"], True),
        ("availp",  ["availability_p"], False),
        ("availq",  ["availability_q"], False),
        ("permp",   ["permutation_p"], False),
        ("permn",   ["permutation_valid_n"], False),
        ("tested",  ["tested"], False),
    ]}
    TESTED = PRIMARY[PRIMARY[P["tested"]].astype(bool)].copy() if P["tested"] else PRIMARY.copy()
except Exception as exc:                                            # noqa: BLE001
    TESTED = None
    record("C00", "primary results table loadable", "FAIL",
           f"{type(exc).__name__}: {exc}")

print("primary rows:", None if PRIMARY is None else len(PRIMARY),
      "| tested:", None if TESTED is None else len(TESTED))

In [ ]:
# =============================================================================
# CHECK 6 - every tested endpoint meets the 6/6 evaluability and prevalence rules
# =============================================================================
@run_check("C06", "every tested endpoint meets the 6/6 evaluability and prevalence rules")
def _c06():
    assert TESTED is not None, "primary results table not loaded"
    bad_g = int((TESTED[P["ng"]] < MIN_EVALUABLE_PER_GROUP).sum())
    bad_p = int((TESTED[P["np_"]] < MIN_EVALUABLE_PER_GROUP).sum())
    ev = {"n_tested": int(len(TESTED)),
          "min_evaluable_per_group": MIN_EVALUABLE_PER_GROUP,
          "min_nonzero_total": MIN_NONZERO_TOTAL,
          "violations_n_good": bad_g, "violations_n_poor": bad_p}
    problems = []
    if bad_g or bad_p:
        problems.append(f"{bad_g} endpoints with n_good < 6, {bad_p} with n_poor < 6")
    if P["nzg"] and P["nzp"]:
        nz = TESTED[P["nzg"]].fillna(0) + TESTED[P["nzp"]].fillna(0)
        bad_nz = int((nz < MIN_NONZERO_TOTAL).sum())
        ev["violations_nonzero_total"] = bad_nz
        if bad_nz:
            problems.append(f"{bad_nz} endpoints with fewer than 6 non-zero scores")
    else:
        ev["violations_nonzero_total"] = "nonzero columns absent"
    if problems:
        return "FAIL", "; ".join(problems), ev
    return "PASS", f"all {len(TESTED)} tested endpoints satisfy 6/6 and prevalence", ev

In [ ]:
# =============================================================================
# CHECK 7 - BH applied to the complete estimable family, once per family
# =============================================================================
@run_check("C07a", "BH applied once across the complete tested primary family")
def _c07a():
    assert TESTED is not None
    _, q_recomputed, _, _ = multipletests(TESTED[P["p"]].values, method=FDR_METHOD)
    maxdiff = float(np.nanmax(np.abs(q_recomputed - TESTED[P["q"]].values)))
    ev = {"family_size": int(len(TESTED)), "method": FDR_METHOD,
          "max_abs_difference_vs_recomputed": maxdiff,
          "n_q_lt_alpha": int((TESTED[P["q"]] < FDR_ALPHA).sum())}
    if P["tested"]:
        untested = PRIMARY[~PRIMARY[P["tested"]].astype(bool)]
        n_untested_with_q = int(untested[P["q"]].notna().sum())
        ev["n_untested_rows"] = int(len(untested))
        ev["n_untested_carrying_q"] = n_untested_with_q
        if n_untested_with_q:
            return "FAIL", (f"{n_untested_with_q} untested endpoints carry a q-value: "
                            "the BH family is not exactly the tested family"), ev
    if maxdiff > 1e-9:
        return "FAIL", (f"stored q-values differ from BH recomputed over the tested family "
                        f"(max abs diff {maxdiff:.2e}): BH was not applied once to this family"), ev
    return "PASS", (f"BH over all {len(TESTED)} tested endpoints reproduces the stored "
                    f"q-values exactly (max diff {maxdiff:.1e})"), ev


@run_check("C07b", "availability tests form a separate BH family")
def _c07b():
    assert TESTED is not None
    if not (P["availp"] and P["availq"]):
        return "WARN", "availability p/q columns absent from the primary table", None
    sub = PRIMARY[PRIMARY[P["availp"]].notna()]
    _, q_re, _, _ = multipletests(sub[P["availp"]].values, method=FDR_METHOD)
    maxdiff = float(np.nanmax(np.abs(q_re - sub[P["availq"]].values)))
    ev = {"availability_family_size": int(len(sub)),
          "max_abs_difference_vs_recomputed": maxdiff,
          "n_availability_q_lt_alpha": int((sub[P["availq"]] < FDR_ALPHA).sum())}
    if maxdiff > 1e-9:
        return "FAIL", (f"availability q-values are not a single BH correction over their "
                        f"own family (max diff {maxdiff:.2e})"), ev
    return "PASS", (f"availability family of {len(sub)} tests BH-corrected once, "
                    "separately from the score family"), ev


@run_check("C07c", "pooled Wilcoxon tables each BH-corrected once within their own file")
def _c07c():
    rep = DE_RESULTS_DIR / "multiplicity_family_report.csv"
    assert rep.is_file(), f"{rep} not found - run 05_ first"
    r = pd.read_csv(rep)
    problems = []
    for _, row in r.iterrows():
        f = DE_RESULTS_DIR / row["table"]
        if not f.is_file():
            problems.append(f"missing {row['table']}")
            continue
        df = pd.read_csv(f)
        if bool(row["bh_corrected"]):
            if "fdr" not in df.columns:
                problems.append(f"{row['table']}: no fdr column")
                continue
            _, q_re, _, _ = multipletests(df["pvalue"].values, method=FDR_METHOD)
            if float(np.nanmax(np.abs(q_re - df["fdr"].values))) > 1e-9:
                problems.append(f"{row['table']}: fdr is not a single BH over its own rows")
        else:
            if "fdr" in df.columns:
                problems.append(f"{row['table']}: uncorrected table carries an fdr column")
    ev = {"families": r.to_dict("records")}
    if problems:
        return "FAIL", "; ".join(problems), ev
    return "PASS", (f"{int(r['bh_corrected'].sum())} pooled families BH-corrected once each; "
                    f"{int((~r['bh_corrected'].astype(bool)).sum())} table(s) explicitly uncorrected"), ev

In [ ]:
# =============================================================================
# CHECK 8 - pooled significance was NOT used to define the primary family
# =============================================================================
@run_check("C08", "pooled significance did not define the primary tested family")
def _c08():
    assert TESTED is not None
    pooled = DE_RESULTS_DIR / "wilcoxon_per_interaction_scores.csv"
    assert pooled.is_file(), f"{pooled} not found - run 05_ first"
    pdf = pd.read_csv(pooled)
    pooled_hit_ids = set(pdf.loc[pdf["fdr"] < FDR_ALPHA, "id_cp_interaction"])
    pooled_tested_ids = set(pdf["id_cp_interaction"])
    tested_ids = set(TESTED[P["id"]])

    n_outside_hits = len(tested_ids - pooled_hit_ids)
    n_outside_pooled = len(tested_ids - pooled_tested_ids)
    ev = {"n_primary_tested_interactions": len(tested_ids),
          "n_pooled_hit_interactions": len(pooled_hit_ids),
          "n_primary_tested_not_pooled_hits": n_outside_hits,
          "n_primary_tested_not_even_pooled_tested": n_outside_pooled,
          "primary_family_rule": "6 evaluable Good + 6 evaluable Poor + >=6 non-zero scores"}

    if pooled_hit_ids and tested_ids.issubset(pooled_hit_ids):
        return "FAIL", ("every primary endpoint belongs to a pooled hit: the primary "
                        "family appears to have been filtered by pooled significance"), ev
    if n_outside_hits == 0:
        return "FAIL", "no primary endpoint lies outside the pooled hit list", ev
    return "PASS", (f"{n_outside_hits} of {len(tested_ids)} tested interactions are not "
                    "pooled hits; the primary family is defined by the prespecified "
                    "evaluability and prevalence rules only"), ev

In [ ]:
# =============================================================================
# CHECK 9 - effect-size signs agree with medians, or the tie case is documented
# =============================================================================
@run_check("C09", "effect-size signs agree with medians (ties documented)")
def _c09():
    assert TESTED is not None
    if P["mdiff"]:
        mdiff = TESTED[P["mdiff"]].astype(float)
    else:
        mdiff = TESTED[P["medg"]].astype(float) - TESTED[P["medp"]].astype(float)
    rb = TESTED[P["rb"]].astype(float)

    # sign(rank-biserial) == sign(median difference) is NOT a mathematical identity:
    # rank-biserial compares the full rank distributions (P(Good > Poor)) while the
    # median is a single order statistic, so the two may point opposite ways when the
    # effect is negligible. What a Good-minus-Poor sign-convention bug WOULD produce is
    # disagreement at materially separated endpoints, so that is what is enforced here.
    MATERIAL_RB = 0.5

    tie = np.isclose(mdiff, 0.0) | np.isclose(rb, 0.0)
    disagree = (~tie) & (np.sign(mdiff) != np.sign(rb))
    material = np.abs(rb) >= MATERIAL_RB
    material_dis = disagree & material
    n_tie, n_dis = int(tie.sum()), int(disagree.sum())
    n_mat, n_mat_dis = int(material.sum()), int(material_dis.sum())

    ev = {"n_tested": int(len(TESTED)), "n_sign_disagreements": n_dis,
          "n_tie_or_zero_effect": n_tie,
          "material_rank_biserial_threshold": MATERIAL_RB,
          "n_material_effects": n_mat,
          "n_material_effects_agreeing": n_mat - n_mat_dis,
          "n_material_disagreements": n_mat_dis,
          "max_abs_rb_among_disagreements":
              float(np.abs(rb[disagree]).max()) if n_dis else 0.0,
          "min_p_among_disagreements":
              float(TESTED.loc[disagree, P["p"]].min()) if n_dis and P.get("p") else None,
          "tie_note": ("median difference or rank-biserial exactly zero. The rank-biserial "
                       "correlation compares full rank distributions while the median "
                       "compares one order statistic, so the two can legitimately differ "
                       "in the tie case."),
          "convention_note": ("a Good-minus-Poor sign-convention error would flip materially "
                              "separated endpoints; agreement there is the real invariant"),
          "examples": TESTED.loc[disagree, [P["id"], P["sender"], P["recv"],
                                            P["medg"], P["medp"], P["rb"]]]
                              .head(10).to_dict("records") if n_dis else []}
    if n_mat_dis:
        return "FAIL", (f"{n_mat_dis} endpoints with |rank_biserial| >= {MATERIAL_RB} where "
                        "its sign disagrees with the median difference: the Good-minus-Poor "
                        "sign convention is wrong"), ev
    if n_dis:
        return "WARN", (f"signs agree on all {n_mat} materially separated endpoints "
                        f"(|rb| >= {MATERIAL_RB}); {n_dis} negligible-effect endpoints "
                        f"(max |rb| {float(np.abs(rb[disagree]).max()):.3f}) disagree, which "
                        "the rank-biserial/median distinction expects; "
                        f"{n_tie} tied endpoints documented"), ev
    return "PASS", (f"signs agree on all {len(TESTED) - n_tie} non-tied endpoints; "
                    f"{n_tie} tied endpoints documented"), ev


In [ ]:
# =============================================================================
# CHECK 10 - confidence intervals and permutation results within expected bounds
# =============================================================================
@run_check("C10a", "rank-biserial and its bootstrap CI are within expected bounds")
def _c10a():
    assert TESTED is not None
    rb, lo, hi = (TESTED[P["rb"]].astype(float), TESTED[P["rblo"]].astype(float),
                  TESTED[P["rbhi"]].astype(float))
    tol = 1e-9
    out_of_range = int(((rb < -1 - tol) | (rb > 1 + tol)).sum())
    ci_out = int(((lo < -1 - tol) | (hi > 1 + tol)).sum())
    inverted = int((lo > hi + tol).sum())
    not_covering = int(((rb < lo - 1e-6) | (rb > hi + 1e-6)).sum())
    nan_ci = int((lo.isna() | hi.isna()).sum())
    ev = {"n_tested": int(len(TESTED)), "rank_biserial_out_of_[-1,1]": out_of_range,
          "ci_bounds_out_of_[-1,1]": ci_out, "ci_low_gt_ci_high": inverted,
          "point_estimate_outside_its_own_CI": not_covering, "nan_ci_bounds": nan_ci}
    problems = [k for k, v in ev.items() if k != "n_tested" and isinstance(v, int) and v > 0
                and k != "point_estimate_outside_its_own_CI"]
    if problems:
        return "FAIL", f"bound violations: {problems}", ev
    if not_covering:
        return "WARN", (f"{not_covering} percentile CIs do not contain the point estimate; "
                        "possible with discrete bootstrap distributions but worth review"), ev
    return "PASS", "all effect sizes and CIs inside [-1, 1] with low <= estimate <= high", ev


@run_check("C10b", "permutation p-values within expected bounds")
def _c10b():
    assert TESTED is not None
    if not P["permp"]:
        return "WARN", "permutation_p column absent from the primary table", None
    sub = TESTED[TESTED[P["permp"]].notna()]
    if len(sub) == 0:
        return "WARN", "no endpoint carries a permutation p-value", {"n": 0}
    pp = sub[P["permp"]].astype(float)
    ev = {"n_with_permutation_p": int(len(sub)), "min": float(pp.min()),
          "max": float(pp.max()),
          "estimator": "(extreme + 1) / (valid + 1)"}
    problems = []
    if ((pp <= 0) | (pp > 1)).any():
        problems.append("permutation p outside (0, 1]")
    if P["permn"]:
        n_valid = sub[P["permn"]].astype(float)
        floor = 1.0 / (n_valid + 1.0)
        ev["n_below_attainable_floor"] = int((pp < floor - 1e-12).sum())
        ev["min_valid_permutations"] = float(n_valid.min())
        if ev["n_below_attainable_floor"]:
            problems.append(f"{ev['n_below_attainable_floor']} permutation p below "
                            "1/(valid+1), the smallest attainable value")
    if problems:
        return "FAIL", "; ".join(problems), ev
    return "PASS", (f"{len(sub)} permutation p-values in "
                    f"[{pp.min():.3g}, {pp.max():.3g}], all at or above their floor"), ev

In [ ]:
# =============================================================================
# CHECK 11 - all output tables have unique keys
# =============================================================================
KEY_SPECS = {
    "patient_level_primary_results.csv": ["id_cp_interaction", "sender", "receiver"],
    "primary_results.csv":               ["id_cp_interaction", "sender", "receiver"],
    "wilcoxon_per_interaction_scores.csv": ["id_cp_interaction"],
    "wilcoxon_per_interaction_means.csv":  ["id_cp_interaction"],
    "wilcoxon_per_cellpair_scores.csv":    ["cell_type_pair"],
    "wilcoxon_per_cellpair_means.csv":     ["cell_type_pair"],
    "detailed_interaction_summary.csv":    ["id_cp_interaction", "cell_type_pair"],
    "consistency_summary.csv":             ["id_cp_interaction"],
    "table1_participant_qc.csv":           ["sample_id"],
    "table2_celltype_qc.csv":              ["cell_type"],
    "table3_patient_celltype_availability.csv": ["sample_id", "cell_type"],
    "table6_availability_bias_flags.csv":  ["id_cp_interaction", "sender", "receiver"],
    "patient_group_assignment.csv":        ["sample_id"],
    "pooled_patient_concordance_interaction.csv": ["id_cp_interaction"],
    "pooled_patient_concordance_cellpair.csv":    ["cell_type_pair"],
}


@run_check("C11", "all output tables have unique keys")
def _c11():
    checked, dup_key, dup_rows, skipped = {}, [], [], []
    for p in sorted(OUTPUT_ROOT.rglob("*.csv")):
        if "figure_verification" in p.name or p.name.endswith("_run_log.jsonl"):
            continue
        try:
            df = pd.read_csv(p)
        except Exception as exc:                                     # noqa: BLE001
            skipped.append(f"{p.name}: {exc}")
            continue
        rel = str(p.relative_to(OUTPUT_ROOT))
        keys = KEY_SPECS.get(p.name)
        if keys and all(k in df.columns for k in keys):
            n_dup = int(df.duplicated(subset=keys).sum())
            checked[rel] = {"keys": keys, "n_duplicate_keys": n_dup, "n_rows": int(len(df))}
            if n_dup:
                dup_key.append(f"{rel}: {n_dup} duplicate {keys}")
        else:
            n_dup = int(df.duplicated().sum())
            checked[rel] = {"keys": "full-row", "n_duplicate_keys": n_dup,
                            "n_rows": int(len(df))}
            if n_dup:
                dup_rows.append(f"{rel}: {n_dup} fully duplicated rows")
    ev = {"n_tables_checked": len(checked), "declared_key_violations": dup_key,
          "full_row_duplicates": dup_rows, "unreadable": skipped, "detail": checked}
    if dup_key:
        return "FAIL", "; ".join(dup_key[:10]), ev
    if dup_rows:
        return "WARN", (f"{len(dup_rows)} tables have fully duplicated rows but no "
                        "declared key: " + "; ".join(dup_rows[:5])), ev
    return "PASS", f"{len(checked)} tables checked, no duplicate keys", ev

In [ ]:
# =============================================================================
# CHECK 12 - figures regenerate from saved tables
# =============================================================================
@run_check("C12", "every figure is traceable to a saved table that still exists")
def _c12():
    prov = FIG_ROOT / "figure_provenance.csv"
    assert prov.is_file(), f"{prov} not found - run 07_ first"
    df = pd.read_csv(prov)
    assert len(df) > 0, "figure provenance table is empty"

    missing_fig, missing_src, stale = [], [], []
    for _, r in df.iterrows():
        fig = OUTPUT_ROOT / str(r["figure"])
        if not fig.is_file():
            missing_fig.append(str(r["figure"]))
            continue
        for src in str(r["source_tables"]).split(";"):
            src = src.strip()
            if not src:
                continue
            sp = Path(src)
            if not sp.is_absolute():
                sp = OUTPUT_ROOT / sp
            if sp.is_dir():
                continue
            if not sp.exists():
                missing_src.append(f"{r['figure']} <- {src}")
                continue
            if sp.stat().st_mtime > fig.stat().st_mtime + 1:
                stale.append(f"{r['figure']} is older than its source {src}")

    figs_on_disk = {str(p.relative_to(OUTPUT_ROOT))
                    for p in list(FIG_ROOT.rglob("*.png")) + list(FIG_ROOT.rglob("*.svg"))}
    unrecorded = sorted(figs_on_disk - set(df["figure"].astype(str)))
    ev = {"n_provenance_rows": int(len(df)), "n_figures_on_disk": len(figs_on_disk),
          "missing_figures": missing_fig[:20], "missing_source_tables": missing_src[:20],
          "figures_older_than_source": stale[:20],
          "figures_without_provenance": unrecorded[:20]}
    problems = []
    if missing_fig:
        problems.append(f"{len(missing_fig)} recorded figures missing from disk")
    if missing_src:
        problems.append(f"{len(missing_src)} figures reference a source table that is gone")
    if stale:
        problems.append(f"{len(stale)} figures predate their source table")
    if unrecorded:
        problems.append(f"{len(unrecorded)} figures have no provenance entry")
    if problems:
        return "FAIL", "; ".join(problems), ev
    return "PASS", (f"all {len(figs_on_disk)} figures trace to an existing saved table "
                    "and postdate it"), ev

In [ ]:
# =============================================================================
# CHECK 13 - config, database, inputs, code and output checksums recorded
# =============================================================================
@run_check("C13", "config, database, inputs, code and output checksums recorded")
def _c13():
    entries, missing = {}, []

    entries["config.yaml"] = CFG_SHA

    db = CFG.get("cpdb_database") if hasattr(CFG, "get") else None
    if db and Path(db).is_file():
        entries["cpdb_database"] = sha256_file(Path(db))
    else:
        missing.append(f"cpdb_database ({db})")

    h5 = CFG.get("input_h5ad") if hasattr(CFG, "get") else None
    if h5 and Path(h5).is_file():
        entries["input_h5ad"] = sha256_file(Path(h5))
    else:
        missing.append(f"input_h5ad ({h5})")

    code_files = sorted(list(CODE_DIR.glob("*.ipynb")) + list(CODE_DIR.glob("*.py")))
    for f in code_files:
        entries[str(f.relative_to(CODE_DIR))] = sha256_file(f)

    outputs = sorted(p for p in OUTPUT_ROOT.rglob("*")
                     if p.is_file() and p.suffix not in (".h5ad",)
                     and p.name != MANIFEST_PATH.name)
    for f in outputs:
        entries[str(f.relative_to(OUTPUT_ROOT))] = sha256_file(f)

    global MANIFEST_ENTRIES
    MANIFEST_ENTRIES = entries

    ev = {"n_entries": len(entries), "n_code_files": len(code_files),
          "n_output_files": len(outputs), "missing": missing}
    if missing:
        return "FAIL", f"cannot checksum: {missing}", ev
    return "PASS", (f"{len(entries)} checksums recorded "
                    f"(config, database, H5AD, {len(code_files)} code files, "
                    f"{len(outputs)} outputs)"), ev


@run_check("C16", "pooled balanced design matches the > 10 retention rule")
def _c16():
    '''The pooled runs must contain exactly the 15 cell types the
    min(n_good, n_poor) > 10 rule retains, and none of the 4 it drops.'''
    found = {}
    for grp in ("good", "poor"):
        sm = POOLED_DIR / grp / "significant_means.csv"
        assert sm.is_file(), f"missing {sm}"
        cols = pd.read_csv(sm, nrows=0).columns
        found[grp] = sorted({ct for c in cols if "|" in c for ct in c.split("|")})

    ev = {"n_types_good": len(found["good"]), "n_types_poor": len(found["poor"]),
          "expected_retained": POOLED_EXPECTED_RETAINED_TYPES,
          "rule": f"min(n_good, n_poor) > {CFG['pooled_celltype_min_per_group']} (strict)",
          "expected_dropped": sorted(POOLED_DROPPED_TYPES),
          "types": found["good"]}

    problems = []
    if found["good"] != found["poor"]:
        problems.append("pooled Good and Poor runs contain different cell-type sets")
    for grp in ("good", "poor"):
        if len(found[grp]) != POOLED_EXPECTED_RETAINED_TYPES:
            problems.append(f"pooled {grp}: {len(found[grp])} cell types, "
                            f"expected {POOLED_EXPECTED_RETAINED_TYPES}")
        present_but_dropped = sorted(set(found[grp]) & set(POOLED_DROPPED_TYPES))
        if present_but_dropped:
            problems.append(f"pooled {grp} contains types the > "
                            f"{CFG['pooled_celltype_min_per_group']} rule drops: "
                            f"{present_but_dropped}")
            ev["present_but_should_be_dropped"] = present_but_dropped

    # Cell counts, if the pooled input meta.tsv can be located.
    counts = {}
    for grp in ("good", "poor"):
        meta = find_first([f"**/pooled/{grp}/meta.tsv", f"**/pooled_seed0/{grp}/meta.tsv"],
                          roots=[OUTPUT_ROOT])
        if meta is not None:
            counts[grp] = int(len(pd.read_csv(meta, sep="\t")))
    if counts:
        ev["cells_per_group"] = counts
        ev["expected_cells_per_group"] = POOLED_EXPECTED_CELLS_PER_GROUP
        for grp, n in counts.items():
            if n != POOLED_EXPECTED_CELLS_PER_GROUP:
                problems.append(f"pooled {grp}: {n:,} cells, "
                                f"expected {POOLED_EXPECTED_CELLS_PER_GROUP:,}")
    else:
        ev["cells_per_group"] = "pooled meta.tsv not located; cell counts not checked"

    if problems:
        return "FAIL", "; ".join(problems), ev
    detail = (f"{POOLED_EXPECTED_RETAINED_TYPES} retained types in both groups, "
              f"{len(POOLED_DROPPED_TYPES)} correctly absent")
    if counts:
        detail += f", {POOLED_EXPECTED_CELLS_PER_GROUP:,} cells per group"
    return "PASS", detail, ev


@run_check("C14", "config.yaml still carries the prespecified design values")
def _c14():
    '''Every notebook derives its thresholds from config.yaml, so a config edit
    would be honoured silently rather than caught. This check pins config.yaml
    against the values locked at prespecification.'''
    mismatches, missing = {}, []
    for k, want in PRESPECIFIED_DESIGN.items():
        got = CFG.get(k)
        if got is None:
            missing.append(k)
        elif list(got) != list(want) if isinstance(want, list) else str(got) != str(want):
            mismatches[k] = {"config": got, "prespecified": want}

    ev = {"prespecified_design": PRESPECIFIED_DESIGN,
          "mismatches": mismatches, "missing_keys": missing,
          "config_sha256": CFG_SHA,
          "note": ("05_, 07_ and 08_ read every analytical threshold from "
                   "config.yaml; this check is what makes that safe.")}
    if missing:
        return "FAIL", f"config.yaml is missing prespecified keys: {missing}", ev
    if mismatches:
        return "FAIL", (f"config.yaml has drifted from the prespecified design: "
                        f"{mismatches}"), ev
    return "PASS", (f"all {len(PRESPECIFIED_DESIGN)} prespecified design values "
                    "still present and unchanged in config.yaml"), ev

In [ ]:
# =============================================================================
# CHECK 17 - interaction-coverage artifact is present, matches the database
#            actually used, and bounds the observed interaction count
# =============================================================================
@run_check("C17", "CellPhoneDB interaction coverage recorded and internally consistent")
def _c17():
    assert COVERAGE_JSON.is_file(), (
        f"{COVERAGE_JSON} not found. Regenerate it with "
        "cpdb_interaction_coverage.py before verification.")
    assert DETECTABILITY_CSV.is_file(), f"{DETECTABILITY_CSV} not found"
    cov = json.loads(COVERAGE_JSON.read_text())

    ia = cov["interactions"]
    ev = {"coverage_json": str(COVERAGE_JSON),
          "coverage_sha256": sha256_file(COVERAGE_JSON),
          "matrix_n_genes": cov["matrix"]["n_genes"],
          "matrix_is_hvg_subset": cov["matrix"]["is_hvg_subset"],
          "interactions": ia,
          "recorded_database_sha256": cov["database"]["sha256"]}
    problems = []

    # (a) the artifact describes the database this analysis actually configured
    db_path = Path(CFG["cpdb_database"])
    if not db_path.is_file():
        problems.append(f"configured cpdb_database not found: {db_path}")
    else:
        db_sha = sha256_file(db_path)
        ev["configured_database_sha256"] = db_sha
        if db_sha != cov["database"]["sha256"]:
            problems.append(
                "coverage artifact was computed against a different CellPhoneDB "
                f"database (recorded {cov['database']['sha256'][:12]}..., configured "
                f"{db_sha[:12]}...); the coverage numbers do not describe this run")

    # (b) the artifact's own arithmetic
    parts = ia["both_partners_present"] + ia["exactly_one_present"] + ia["neither_present"]
    if parts != ia["total"]:
        problems.append(f"coverage partition sums to {parts}, not total {ia['total']}")

    det = pd.read_csv(DETECTABILITY_CSV)
    n_det = int(det["detectable"].astype(bool).sum())
    ev["detectability_rows"] = int(len(det))
    ev["detectability_true"] = n_det
    if len(det) != ia["total"]:
        problems.append(f"detectability table has {len(det)} rows, expected {ia['total']}")
    if n_det != ia["both_partners_present"]:
        problems.append(f"detectability table flags {n_det} detectable, "
                        f"coverage reports {ia['both_partners_present']}")
    if det["id_cp_interaction"].duplicated().any():
        problems.append("detectability table has duplicate id_cp_interaction values")

    # (c) THE INVARIANT: a run cannot report more interactions than are
    #     representable on this matrix. More would mean something is wrong.
    obs = {}
    for label, d in [("all_samples", ALL_SAMPLES_DIR),
                     ("pooled_good", POOLED_DIR / "good"),
                     ("pooled_poor", POOLED_DIR / "poor")]:
        f = d / "interaction_scores.csv"
        if not f.is_file():
            continue
        n_unique = int(pd.read_csv(f, usecols=["id_cp_interaction"])
                         ["id_cp_interaction"].nunique())
        obs[label] = n_unique
        if n_unique > ia["both_partners_present"]:
            problems.append(
                f"{label} run reports {n_unique} distinct interactions, exceeding the "
                f"{ia['both_partners_present']} representable on a "
                f"{cov['matrix']['n_genes']}-gene matrix")
    ev["observed_distinct_interactions"] = obs or "no run outputs available yet"

    # (d) CONTRADICTION CHECK: a classification with zero detectable interactions
    #     cannot produce a result row at all. If one is ever reported as
    #     significant, the pipeline is joining against the wrong database, a
    #     stale file, or a mis-keyed merge -- not a borderline finding.
    cbc = coverage_by_classification()
    zero_cls = set(cbc.loc[cbc["detectable"] == 0, "classification"])
    ev["n_zero_coverage_classifications"] = len(zero_cls)
    ev["zero_coverage_interactions"] = int(
        cbc.loc[cbc["detectable"] == 0, "total"].sum())

    reported = {}
    _detail = DE_RESULTS_DIR / "detailed_interaction_summary.csv"
    if _detail.is_file():
        df = pd.read_csv(_detail)
        if "classification" in df.columns:
            reported["detailed_interaction_summary"] = set(
                df["classification"].dropna().astype(str).str.strip())
    for _f in ("wilcoxon_per_interaction_scores.csv",
               "wilcoxon_per_interaction_means.csv"):
        p = DE_RESULTS_DIR / _f
        if p.is_file():
            df = pd.read_csv(p)
            if {"classification", "fdr"} <= set(df.columns):
                reported[_f] = set(df.loc[df["fdr"] < FDR_ALPHA, "classification"]
                                   .dropna().astype(str).str.strip())
    _cons = CONSISTENCY_DIR / "consistency_summary.csv"
    if _cons.is_file():
        df = pd.read_csv(_cons)
        if "classification" in df.columns:
            reported["consistency_summary"] = set(
                df["classification"].dropna().astype(str).str.strip())

    contradictions = {src: sorted(cls & zero_cls) for src, cls in reported.items()
                      if cls & zero_cls}
    ev["result_tables_checked"] = sorted(reported) or "none available yet"
    if contradictions:
        ev["contradictions"] = contradictions
        problems.append(
            "a classification with ZERO detectable interactions is reported as "
            f"significant: {contradictions}. This is arithmetically impossible on "
            "this matrix and indicates a pipeline fault, not a finding")

    if problems:
        return "FAIL", "; ".join(problems), ev
    detail = (f"{ia['both_partners_present']}/{ia['total']} interactions evaluable "
              f"({100 * ia['detectable_fraction']:.1f}%) on a "
              f"{cov['matrix']['n_genes']:,}-gene matrix; database hash matches; "
              f"{len(zero_cls)} zero-coverage classifications "
              f"({ev['zero_coverage_interactions']} interactions) absent from all "
              "result tables")
    if obs:
        detail += f"; observed {obs} within bound"
    return "PASS", detail, ev

In [ ]:
# =============================================================================
# Write the final MANIFEST.sha256
# =============================================================================
manifest_lines = []
for name in sorted(MANIFEST_ENTRIES):
    manifest_lines.append(f"{MANIFEST_ENTRIES[name]}  {name}")
MANIFEST_PATH.write_text("\n".join(manifest_lines) + "\n")

assert MANIFEST_PATH.is_file() and MANIFEST_PATH.stat().st_size > 0
print(f"MANIFEST.sha256: {len(manifest_lines)} entries -> {MANIFEST_PATH}")

In [ ]:
# =============================================================================
# Collect the computed values used to populate the written documents.
# Only values actually computed here are inserted; nothing is invented.
# =============================================================================
V = {}
V["n_patients"] = EXPECTED_N_PATIENTS
V["n_good"] = EXPECTED_N_GOOD
V["n_poor"] = EXPECTED_N_POOR
V["n_celltypes"] = EXPECTED_N_CELLTYPES
V["fdr_alpha"] = FDR_ALPHA
V["min_evaluable"] = MIN_EVALUABLE_PER_GROUP
V["min_nonzero"] = MIN_NONZERO_TOTAL

if TESTED is not None:
    V["n_endpoint_rows"] = int(len(PRIMARY))
    V["n_tested"] = int(len(TESTED))
    V["n_q_lt_alpha"] = int((TESTED[P["q"]] < FDR_ALPHA).sum())
    V["min_q"] = float(TESTED[P["q"]].min())
    V["min_p"] = float(TESTED[P["p"]].min())
    V["rb_min"] = float(TESTED[P["rb"]].min())
    V["rb_max"] = float(TESTED[P["rb"]].max())
    V["n_availability_flagged"] = (int((PRIMARY[P["availq"]] < FDR_ALPHA).sum())
                                   if P["availq"] else None)
    top = TESTED.nsmallest(10, P["q"])
    V["top10"] = top[[c for c in [P["id"], P["sender"], P["recv"], P["ng"], P["np_"],
                                  P["medg"], P["medp"], P["rb"], P["rblo"], P["rbhi"],
                                  P["p"], P["q"]] if c]].to_dict("records")
else:
    V["n_tested"] = None
    V["top10"] = []

try:
    _fam = pd.read_csv(DE_RESULTS_DIR / "multiplicity_family_report.csv")
    V["pooled_families"] = _fam.to_dict("records")
except Exception:
    V["pooled_families"] = []

try:
    _cs = pd.read_csv(CONSISTENCY_DIR / "consistency_summary.csv")
    V["n_consistency_checked"] = int(len(_cs))
    V["n_consistency_consistent"] = int(_cs["consistent_with_pooled"].sum())
except Exception:
    V["n_consistency_checked"] = None
    V["n_consistency_consistent"] = None

try:
    _ci = pd.read_csv(EXPLORATORY_DIR / "pooled_patient_concordance_interaction.csv")
    V["concordance_n"] = int(len(_ci))
    V["concordance_frac"] = float(_ci["direction_concordant"].mean())
except Exception:
    V["concordance_n"] = None
    V["concordance_frac"] = None

# --- Interaction coverage, read from the audit artifact (never retyped) -----
V["coverage"] = None
V["coverage_by_class"] = None
try:
    V["coverage"] = json.loads(COVERAGE_JSON.read_text())
    V["coverage_path"] = str(COVERAGE_JSON)
    # Per-classification breakdown, grouped from the per-interaction artifact.
    V["coverage_by_class"] = coverage_by_classification()
    _missing_acc = [c for c in ACC_COVERAGE_CLASSIFICATIONS
                    if c not in set(V["coverage_by_class"]["classification"])]
    assert not _missing_acc, (
        f"ACC classifications absent from the database: {_missing_acc}. The exact "
        "classification names have changed; update ACC_COVERAGE_CLASSIFICATIONS.")
except Exception as exc:                                            # noqa: BLE001
    record("C17b", "interaction-coverage artifact readable for documents",
           "FAIL", f"{type(exc).__name__}: {exc}")

# --- Site of Extraction x outcome, read from the authoritative covariate CSV --
# config.yaml warns that obs['site'] splits "Base of Tongue" into two labels
# that segregate perfectly by outcome; the CSV column is the authoritative
# source and is what is tabulated here.
V["site_table"] = None
V["site_source"] = None
try:
    cov_path = Path(CFG["covariate_csv"])
    site_col = CFG["covariate_column_map"]["site_of_extraction"]
    cov = pd.read_csv(cov_path)
    gp = find_first(["**/patient_group_assignment.csv", "**/*patient_outcome*.csv",
                     "**/*derived_patient*.csv"])
    grp = read_any(gp)
    g_id = col(grp, ["sample_id", "batch", "sample", "patient", "patient_id"])
    g_oc = col(grp, ["clinical_outcome", "outcome", "group"])
    omap = dict(zip(grp[g_id].astype(str), grp[g_oc].astype(str)))
    c_id = col(cov, ["Sample ID", "sample_id", "batch", "sample"])
    cov = cov[cov[c_id].astype(str).isin(omap)].copy()
    cov["clinical_outcome"] = cov[c_id].astype(str).map(omap)
    tab = (pd.crosstab(cov[site_col], cov["clinical_outcome"])
             .reindex(columns=["Good", "Poor"], fill_value=0)
             .sort_values(["Poor", "Good"], ascending=False))
    assert int(tab["Good"].sum()) == EXPECTED_N_GOOD
    assert int(tab["Poor"].sum()) == EXPECTED_N_POOR
    # Rows named in the analysis specification must match the data exactly.
    SPEC_ROWS = {"Maxilla": (0, 4), "Larynx": (2, 0), "Tongue": (2, 0),
                 "Unknown": (4, 1)}
    spec_mismatch = {k: (int(tab.loc[k, "Good"]), int(tab.loc[k, "Poor"]))
                     for k in SPEC_ROWS if k in tab.index
                     and (int(tab.loc[k, "Good"]), int(tab.loc[k, "Poor"])) != SPEC_ROWS[k]}
    assert not spec_mismatch, f"site table disagrees with the specification: {spec_mismatch}"
    V["site_table"] = tab
    V["site_source"] = str(cov_path)
    record("C15", "Site of Extraction imbalance tabulated from the covariate CSV",
           "PASS", f"{len(tab)} site labels, 13 Good / 11 Poor, spec rows match",
           {"source": str(cov_path), "table": tab.to_dict()})
except Exception as exc:                                            # noqa: BLE001
    record("C15", "Site of Extraction imbalance tabulated from the covariate CSV",
           "WARN", f"{type(exc).__name__}: {exc}")

n_pass = sum(1 for c in CHECKS if c["status"] == "PASS")
n_warn = sum(1 for c in CHECKS if c["status"] == "WARN")
n_fail = sum(1 for c in CHECKS if c["status"] == "FAIL")
V["n_pass"], V["n_warn"], V["n_fail"] = n_pass, n_warn, n_fail

checks_df = pd.DataFrame(CHECKS)
checks_df.to_csv(VERIFICATION_CSV, index=False)
checks_df[["check_id", "name", "status", "detail"]]

In [ ]:
# =============================================================================
# FINAL_VERIFICATION.md
# =============================================================================
def _fmt(x, nd=4):
    if x is None:
        return "not computed"
    if isinstance(x, float):
        return f"{x:.{nd}g}"
    return str(x)


lines = []
lines.append("# Final verification — CellPhoneDB v5 clinical-outcome re-analysis")
lines.append("")
lines.append(f"Generated: {datetime.now(timezone.utc).isoformat()}")
lines.append(f"Output root: `{OUTPUT_ROOT}`")
lines.append(f"Config SHA-256: `{CFG_SHA}`")
lines.append(f"Workflow start used for modification checks: {WORKFLOW_START_FIND}")
lines.append("")
lines.append(f"**{n_pass} PASS · {n_warn} WARN · {n_fail} FAIL**")
lines.append("")
if n_fail == 0:
    lines.append("No unresolved failures. The workflow is complete.")
else:
    lines.append("**Unresolved failures are present. The workflow is NOT complete.**")
lines.append("")
lines.append("## Check results")
lines.append("")
lines.append("| ID | Check | Status | Detail |")
lines.append("|---|---|---|---|")
for c in CHECKS:
    detail = c["detail"].replace("|", "\\|").replace("\n", " ")
    lines.append(f"| {c['check_id']} | {c['name']} | **{c['status']}** | {detail} |")
lines.append("")

if n_fail:
    lines.append("## Failures in detail")
    lines.append("")
    for c in CHECKS:
        if c["status"] == "FAIL":
            lines.append(f"### {c['check_id']} — {c['name']}")
            lines.append("")
            lines.append(c["detail"])
            lines.append("")
            if c["evidence"]:
                lines.append("```json")
                lines.append(c["evidence"][:4000])
                lines.append("```")
                lines.append("")

if n_warn:
    lines.append("## Warnings")
    lines.append("")
    for c in CHECKS:
        if c["status"] == "WARN":
            lines.append(f"- **{c['check_id']} {c['name']}** — {c['detail']}")
    lines.append("")

lines.append("## Recorded quantities")
lines.append("")
lines.append(f"- Patients: {V['n_patients']} ({V['n_good']} Good, {V['n_poor']} Poor), "
             f"cell types: {V['n_celltypes']}")
lines.append(f"- CellPhoneDB runs expected: {EXPECTED_TOTAL_RUNS} "
             f"({EXPECTED_CORE_RUNS} core + {EXPECTED_SEED_RUNS} pooled seed)")
lines.append(f"- Primary endpoint rows: {_fmt(V.get('n_endpoint_rows'))}; "
             f"tested (estimable) endpoints: {_fmt(V.get('n_tested'))}")
lines.append(f"- Endpoints with Benjamini–Hochberg q < {FDR_ALPHA}: {_fmt(V.get('n_q_lt_alpha'))}")
lines.append(f"- Smallest q-value: {_fmt(V.get('min_q'))}; smallest raw p-value: {_fmt(V.get('min_p'))}")
lines.append(f"- Rank-biserial range across tested endpoints: "
             f"{_fmt(V.get('rb_min'))} to {_fmt(V.get('rb_max'))}")
lines.append(f"- Endpoints flagged for differential availability (q < {FDR_ALPHA}): "
             f"{_fmt(V.get('n_availability_flagged'))}")
lines.append(f"- Exploratory pooled consistency: {_fmt(V.get('n_consistency_consistent'))} of "
             f"{_fmt(V.get('n_consistency_checked'))} pooled hits agree in direction across patients")
lines.append(f"- Pooled↔patient direction concordance (annotation only): "
             f"{_fmt(V.get('concordance_frac'))} over {_fmt(V.get('concordance_n'))} aligned keys")
if V.get("coverage"):
    _ia = V["coverage"]["interactions"]
    lines.append(f"- Testable interaction space: {_ia['both_partners_present']:,} of "
                 f"{_ia['total']:,} database interactions "
                 f"({100 * _ia['detectable_fraction']:.1f} %) on a "
                 f"{V['coverage']['matrix']['n_genes']:,}-gene matrix")
lines.append(f"- Manifest entries: {len(MANIFEST_ENTRIES)} (`MANIFEST.sha256`)")
lines.append("")
lines.append("## Companion files")
lines.append("")
lines.append("- `MANIFEST.sha256`")
lines.append("- `final_verification_checks.csv` (machine-readable)")
lines.append("")

VERIFICATION_MD.write_text("\n".join(lines))
print(f"wrote {VERIFICATION_MD}  ({n_pass} PASS / {n_warn} WARN / {n_fail} FAIL)")

In [ ]:
# =============================================================================
# SUCCESS RECORD + final gate
# =============================================================================
DOCS = [VERIFICATION_MD, MANIFEST_PATH, VERIFICATION_CSV]
for d in DOCS:
    assert d.is_file() and d.stat().st_size > 0, f"document not written: {d}"

SUCCESS = {
    "notebook": NOTEBOOK,
    "status": "success" if n_fail == 0 else "failed_checks_present",
    "run_start_utc": RUN_START_ISO,
    "run_end_utc": datetime.now(timezone.utc).isoformat(),
    "elapsed_seconds": round(time.time() - RUN_START, 2),
    "config_sha256": CFG_SHA,
    "workflow_start_utc_used": WORKFLOW_START_FIND,
    "n_pass": n_pass, "n_warn": n_warn, "n_fail": n_fail,
    "checks": CHECKS,
    "documents": [str(d) for d in DOCS],
    "manifest_entries": len(MANIFEST_ENTRIES),
    "output_sha256": {str(d): sha256_file(d) for d in DOCS},
}
SUCCESS_JSON.write_text(json.dumps(SUCCESS, indent=2))
_call_flex(log_jsonl,
           ((RUN_LOG, {"notebook": NOTEBOOK, "event": "notebook_success",
                       **{k: v for k, v in SUCCESS.items() if k != "checks"}}), {}),
           (({"notebook": NOTEBOOK, "event": "notebook_success",
              **{k: v for k, v in SUCCESS.items() if k != "checks"}}, RUN_LOG), {}))

failed = [c["check_id"] + " " + c["name"] for c in CHECKS if c["status"] == "FAIL"]
assert not failed, (
    "FINAL_VERIFICATION.md contains unresolved failures; the workflow is NOT complete:\n  "
    + "\n  ".join(failed))

print(f"\nVERIFICATION COMPLETE — {n_pass} PASS, {n_warn} WARN, {n_fail} FAIL")
print(f"  {VERIFICATION_MD}")
print(f"  {MANIFEST_PATH}  ({len(MANIFEST_ENTRIES)} entries)")